# Formulas and Named Ranges

This tutorial embeds Excel formulas and creates named ranges using XLSX.jl v0.11+. The formulas are stored in the file and evaluated by Excel when the workbook is opened — XLSX.jl does not compute them.

In [ ]:
Pollis.packages("XLSX")

In [ ]:
using XLSX

## Write Data and Embed Formulas

In [ ]:
XLSX.openxlsx("model.xlsx"; mode="w") do xf
    ws = xf[1]

    # Headers
    ws["A1"] = "Month"
    ws["B1"] = "Revenue"
    ws["C1"] = "Costs"
    ws["D1"] = "Profit"

    # Data rows
    months  = ["Jan", "Feb", "Mar"]
    revenue = [12000.0, 15000.0, 11000.0]
    costs   = [8000.0,  9500.0,  7500.0]

    for (i, (m, r, c)) in enumerate(zip(months, revenue, costs))
        row = string(i + 1)
        ws["A" * row] = m
        ws["B" * row] = r
        ws["C" * row] = c
        XLSX.setFormula(ws, "D" * row, "B" * row * "-C" * row)
    end

    # Totals row
    XLSX.setFormula(ws, "B5", "SUM(B2:B4)")
    XLSX.setFormula(ws, "C5", "SUM(C2:C4)")
    XLSX.setFormula(ws, "D5", "SUM(D2:D4)")

    # Named range: point to the total profit cell
    XLSX.addDefinedName(xf, "TotalProfit", "Sheet1!\$D\$5")
end

## Verify the File Structure

Re-open the file and print the raw cell values (before Excel evaluates the formulas).

In [ ]:
xf = XLSX.readxlsx("model.xlsx")
ws = xf[1]
println(ws["A1:D5"])

## Summary
- `setFormula(ws, cell, formula_string)` embeds an Excel formula — use Excel syntax, not Julia.
- `addDefinedName(xf, name, ref)` creates a workbook-scoped named range.
- Formulas are evaluated by Excel when the file is opened, not by XLSX.jl.
- Use `\$` in absolute references inside Julia strings to avoid interpolation.